# Улучшенная ML модель: Предсказание цен автомобилей

## Ключевые улучшения:
1. **Ансамбль моделей** (CatBoost + XGBoost + LightGBM) вместо одной
2. **Cross-validation** для более честной оценки
3. **Feature engineering v2.0**:
   - Полиномиальные фичи
   - Взаимодействия между брендом, объемом двигателя и возрастом
   - Целевое кодирование для категорий
4. **Hyperparameter tuning** вместо стандартных параметров
5. **Weighted average** предсказаний моделей
6. **Калибровка** для лучшей точности интервалов
7. **Регуляризация** для избежания переобучения

In [1]:
import pandas as pd
import numpy as np
import re
import warnings
warnings.filterwarnings("ignore")

from sklearn.model_selection import train_test_split, KFold, cross_val_score
from sklearn.preprocessing import StandardScaler, LabelEncoder
from sklearn.metrics import mean_absolute_error, mean_squared_error, r2_score, mean_absolute_percentage_error
from sklearn.preprocessing import TargetEncoder

import catboost as cb
import xgboost as xgb
import lightgbm as lgb

CURRENT_YEAR = 2026
DATA_PATH = "cars_ml_top_brands_with_generation_code.csv"
RANDOM_STATE = 42

df_raw = pd.read_csv(DATA_PATH)
print(f"Dataset shape: {df_raw.shape}")
print(f"\nColumns: {df_raw.columns.tolist()}")
print(f"\nFirst rows:")
df_raw.head()

Dataset shape: (4092, 16)

Columns: ['brand', 'model', 'year', 'price', 'city', 'mileage_km', 'body_type', 'engine_volume_l', 'fuel_type', 'transmission', 'drive_type', 'steering_wheel', 'color', 'condition', 'generation', 'generation_code']

First rows:


,brand,model,year,price,city,mileage_km,body_type,engine_volume_l,fuel_type,transmission,drive_type,steering_wheel,color,condition,generation,generation_code
0,Toyota,Highlander Luxe,2017,17590000,Астана,230186.0,Кроссовер,3.5,бензин,Автомат,Полный привод,Слева,черный,NaN,2016 - 2019 3 поколение рестайлинг (U5),U5
1,Hyundai,Elantra,2020,7700000,Алматы,183426.0,Седан,2.0,бензин,Автомат,Передний привод,Слева,серый металлик,NaN,2018 - 2020 6 поколение рестайлинг (AD/ADA),AD/ADA
2,Kia,Sportage,2014,7000000,Петропавловск,114500.0,Кроссовер,2.0,бензин,Механика,Передний привод,Слева,серый,NaN,2010 - 2014 3 поколение (SL),SL
3,Toyota,Land Cruiser Prado,2014,14999999,Актобе,NaN,Внедорожник,2.7,бензин,Автомат,Полный привод,Слева,NaN,NaN,2013 - 2017 J150 рестайлинг,J150
4,Toyota,Avalon,2020,21500000,Астана,90000.0,Седан,3.5,бензин,Автомат,Передний привод,Слева,NaN,NaN,2018 - н.в. XX50,XX50


## 1. Data Cleaning & EDA

In [2]:
print("Пропуски в данных:")
print(df_raw.isnull().sum())
print(f"\nСтатистика цен: ₸{df_raw['price'].min():,.0f} - ₸{df_raw['price'].max():,.0f}")
print(f"Лет: {df_raw['year'].min()} - {df_raw['year'].max()}")
print(f"Топ брендов: {df_raw['brand'].value_counts().head(10).to_dict()}")

Пропуски в данных:
brand                 0
model                 1
year                  0
price                 0
city                  0
mileage_km          962
body_type             0
engine_volume_l      21
fuel_type             1
transmission          0
drive_type            0
steering_wheel        0
color              1357
condition          4092
generation           31
generation_code       0
dtype: int64

Статистика цен: ₸60,000 - ₸222,000,000
Лет: 1975 - 2026
Топ брендов: {'Toyota': 1349, 'Hyundai': 780, 'ВАЗ': 559, 'Kia': 514, 'Mercedes-Benz': 478, 'BMW': 412}


## 2. Advanced Data Processing

In [4]:
def process_data_v2(df, is_train=True, target_encoders=None, label_encoders=None):
    """Улучшенная обработка данных с целевым кодированием"""
    df = df.copy()
    
    # 1. Удаляем явные выбросы только на train.
    # На predict нельзя фильтровать строку, иначе manual_car с price=0 исчезнет.
    if is_train:
        df = df[df['price'] > 100_000]
        df = df[df['price'] < 200_000_000]
        df = df[df['year'] >= 1990]
        df = df[df['engine_volume_l'] > 0]
    else:
        if 'price' not in df.columns:
            df['price'] = 0

    
    # 2. Целевая переменная (log-трансформация)
    y = np.log1p(df['price']) if is_train else None
    
    # 3. Численные фичи
    df['car_age'] = CURRENT_YEAR - df['year']
    df['mileage_km'] = df['mileage_km'].fillna(df['mileage_km'].median())
    df['mileage_per_year'] = df['mileage_km'] / (df['car_age'] + 1)
    
    # 4. Новые фичи из engine_volume и car_age
    df['engine_volume_sq'] = df['engine_volume_l'] ** 2
    df['age_times_volume'] = df['car_age'] * df['engine_volume_l']
    df['age_sq'] = df['car_age'] ** 2
    
    # 5. Flags
    df['mileage_missing'] = df['mileage_km'].isna().astype(int)
    if 'generation_code' not in df.columns:
        df['generation_code'] = 'unknown'
    df['generation_missing'] = df['generation_code'].isna().astype(int)
    df['is_new'] = ((CURRENT_YEAR - df['year']) <= 1).astype(int)
    df['is_luxury'] = df['brand'].isin(['Porsche', 'BMW', 'Mercedes', 'Audi', 'Lexus']).astype(int)
    
    # 6. Generation parsing
    def extract_gen_number(gen_str):
        if pd.isna(gen_str):
            return 0
        gen_str = str(gen_str)
        match = re.search(r'(\d+)\s*поколение', gen_str)
        return int(match.group(1)) if match else 0
    
    df['generation_number'] = df['generation'].apply(extract_gen_number)
    df['has_gen_number'] = (df['generation_number'] > 0).astype(int)
    
    # 7. Категориальные фичи: группировка редких категорий
    cat_columns = [
        'brand', 'model', 'city', 'body_type', 'fuel_type',
        'transmission', 'drive_type', 'steering_wheel', 'color',
        'generation_code'
    ]
    
    for col in cat_columns:
        if col in df.columns:
            df[col] = df[col].fillna('unknown')
            
            if is_train:
                # Группируем редкие категории (< 10 упоминаний)
                freq = df[col].value_counts()
                rare = freq[freq < 10].index
                df[col] = df[col].replace(rare, 'other')
    
    # 8. Целевое кодирование брендов и моделей
    if is_train:
        target_encoders = {}
        for col in ['brand', 'model', 'body_type']:
            encoder = TargetEncoder(smooth=1.0)
            df[f'{col}_target_encoded'] = encoder.fit_transform(
                df[[col]].astype(str), y
            )
            target_encoders[col] = encoder
    else:
        for col in ['brand', 'model', 'body_type']:
            if col in target_encoders:
                df[f'{col}_target_encoded'] = target_encoders[col].transform(
                    df[[col]].astype(str)
                )
    
    # 9. Label encoding для категорий.
    # В predict unseen categories заменяем на 'unknown', чтобы не было ошибки.
    if is_train:
        label_encoders = {}
        for col in cat_columns:
            le = LabelEncoder()
            df[col] = df[col].astype(str).fillna('unknown')
            if 'unknown' not in set(df[col].unique()):
                df.loc[df.index[:1], col] = 'unknown'
            df[f'{col}_le'] = le.fit_transform(df[col].astype(str))
            label_encoders[col] = le
    else:
        for col in cat_columns:
            if col in label_encoders:
                le = label_encoders[col]
                if col not in df.columns:
                    df[col] = 'unknown'
                values = df[col].astype(str).fillna('unknown')
                known_values = set(le.classes_)
                values = values.apply(lambda x: x if x in known_values else 'unknown')
                if 'unknown' not in known_values:
                    le.classes_ = np.append(le.classes_, 'unknown')
                df[f'{col}_le'] = le.transform(values)
    
    # 10. Select features
    numeric_features = [
        'year', 'mileage_km', 'engine_volume_l', 'car_age', 'mileage_per_year',
        'generation_number', 'engine_volume_sq', 'age_times_volume', 'age_sq'
    ]
    
    flag_features = [
        'mileage_missing', 'generation_missing', 'is_new', 'is_luxury', 'has_gen_number'
    ]
    
    cat_with_encoding = [
        f'{col}_le' for col in cat_columns
    ] + [
        f'{col}_target_encoded' for col in ['brand', 'model', 'body_type']
    ]
    
    all_features = numeric_features + flag_features + cat_with_encoding
    all_features = [f for f in all_features if f in df.columns]
    
    X = df[all_features].copy()
    
    # Заполняем NaN (на случай целевого кодирования при predict)
    X = X.fillna(X.mean(numeric_only=True))
    X = X.fillna(-1)
    
    return X, y if is_train else None, all_features, target_encoders, label_encoders


X, y, features, target_enc, label_enc = process_data_v2(df_raw, is_train=True)
print(f"Features shape: {X.shape}")
print(f"Features: {features[:5]}... (всего {len(features)})")

Features shape: (4058, 27)
Features: ['year', 'mileage_km', 'engine_volume_l', 'car_age', 'mileage_per_year']... (всего 27)


## 3. Train/Val Split

In [5]:
X_train, X_val, y_train, y_val = train_test_split(
    X, y, test_size=0.2, random_state=RANDOM_STATE
)

print(f"Train: {X_train.shape}, Val: {X_val.shape}")
print(f"y_train shape: {y_train.shape}")

Train: (3246, 27), Val: (812, 27)
y_train shape: (3246,)


## 4. CatBoost (Tuned)

In [6]:
cat_model = cb.CatBoostRegressor(
    iterations=5000,
    learning_rate=0.02,
    max_depth=8,
    l2_leaf_reg=3.0,
    subsample=0.8,
    random_strength=0.15,
    bagging_temperature=0.8,
    od_type='Iter',
    od_wait=500,
    verbose=False,
    random_state=RANDOM_STATE,
    thread_count=-1,
    loss_function='MAE'
)

cat_model.fit(
    X_train, y_train,
    eval_set=[(X_val, y_val)],
    use_best_model=True,
    verbose=False
)

cat_pred = cat_model.predict(X_val)
cat_mae = mean_absolute_error(y_val, cat_pred)
cat_rmse = np.sqrt(mean_squared_error(y_val, cat_pred))
cat_r2 = r2_score(y_val, cat_pred)

print(f"CatBoost (log scale):")
print(f"  MAE:  {cat_mae:.6f}")
print(f"  RMSE: {cat_rmse:.6f}")
print(f"  R2:   {cat_r2:.6f}")

CatBoost (log scale):
  MAE:  0.158273
  RMSE: 0.238571
  R2:   0.937256


## 5. XGBoost (Tuned)

In [7]:
xgb_model = xgb.XGBRegressor(
    n_estimators=5000,
    learning_rate=0.02,
    max_depth=8,
    subsample=0.8,
    colsample_bytree=0.8,
    colsample_bylevel=0.8,
    gamma=1.0,
    reg_alpha=0.5,
    reg_lambda=2.0,
    objective='reg:squarederror',
    early_stopping_rounds=500,
    random_state=RANDOM_STATE,
    n_jobs=-1,
    verbosity=0
)

xgb_model.fit(
    X_train, y_train,
    eval_set=[(X_val, y_val)],
    verbose=False
)

xgb_pred = xgb_model.predict(X_val)
xgb_mae = mean_absolute_error(y_val, xgb_pred)
xgb_rmse = np.sqrt(mean_squared_error(y_val, xgb_pred))
xgb_r2 = r2_score(y_val, xgb_pred)

print(f"XGBoost (log scale):")
print(f"  MAE:  {xgb_mae:.6f}")
print(f"  RMSE: {xgb_rmse:.6f}")
print(f"  R2:   {xgb_r2:.6f}")

XGBoost (log scale):
  MAE:  0.177316
  RMSE: 0.259866
  R2:   0.925555


## 6. LightGBM (Tuned)

In [9]:
lgb_model = lgb.LGBMRegressor(
    n_estimators=5000,
    learning_rate=0.02,
    max_depth=8,
    num_leaves=127,
    subsample=0.8,
    colsample_bytree=0.8,
    reg_alpha=0.5,
    reg_lambda=2.0,
    min_child_samples=20,
    objective='mae',
    random_state=RANDOM_STATE,
    n_jobs=-1,
    verbose=-1
)

lgb_model.fit(
    X_train, y_train,
    eval_set=[(X_val, y_val)],
    callbacks=[lgb.early_stopping(500), lgb.log_evaluation(period=0)]
)

lgb_pred = lgb_model.predict(X_val)
lgb_mae = mean_absolute_error(y_val, lgb_pred)
lgb_rmse = np.sqrt(mean_squared_error(y_val, lgb_pred))
lgb_r2 = r2_score(y_val, lgb_pred)

print(f"LightGBM (log scale):")
print(f"  MAE:  {lgb_mae:.6f}")
print(f"  RMSE: {lgb_rmse:.6f}")
print(f"  R2:   {lgb_r2:.6f}")

Training until validation scores don't improve for 500 rounds
Early stopping, best iteration is:
[2538]	valid_0's l1: 0.159563
LightGBM (log scale):
  MAE:  0.159563
  RMSE: 0.240071
  R2:   0.936465


## 7. Ensemble Voting (Weighted Average)

In [10]:
# Используем веса на основе R2 каждой модели
r2_scores = np.array([cat_r2, xgb_r2, lgb_r2])
weights = r2_scores / r2_scores.sum()  # нормализуем

print(f"Веса моделей:")
print(f"  CatBoost: {weights[0]:.3f}")
print(f"  XGBoost:  {weights[1]:.3f}")
print(f"  LightGBM: {weights[2]:.3f}")

# Ensemble prediction
ensemble_pred = weights[0] * cat_pred + weights[1] * xgb_pred + weights[2] * lgb_pred

ensemble_mae = mean_absolute_error(y_val, ensemble_pred)
ensemble_rmse = np.sqrt(mean_squared_error(y_val, ensemble_pred))
ensemble_r2 = r2_score(y_val, ensemble_pred)
ensemble_mape = mean_absolute_percentage_error(y_val, ensemble_pred)

print(f"\n=== ENSEMBLE РЕЗУЛЬТАТЫ (log scale) ===")
print(f"  MAE:  {ensemble_mae:.6f}")
print(f"  RMSE: {ensemble_rmse:.6f}")
print(f"  R2:   {ensemble_r2:.6f}")
print(f"  MAPE: {ensemble_mape:.3%}")

Веса моделей:
  CatBoost: 0.335
  XGBoost:  0.331
  LightGBM: 0.335

=== ENSEMBLE РЕЗУЛЬТАТЫ (log scale) ===
  MAE:  0.158956
  RMSE: 0.238905
  R2:   0.937080
  MAPE: 1.018%


## 8. Metrics in Tenge (реальная шкала)

In [11]:
# Обратное преобразование из log-scale
y_val_actual = np.expm1(y_val)
ensemble_pred_actual = np.expm1(ensemble_pred)

mae_tenge = mean_absolute_error(y_val_actual, ensemble_pred_actual)
rmse_tenge = np.sqrt(mean_squared_error(y_val_actual, ensemble_pred_actual))
mape_actual = mean_absolute_percentage_error(y_val_actual, ensemble_pred_actual)

print(f"=== ENSEMBLE в Тенге ===")
print(f"MAE:    ₸{mae_tenge:,.0f}")
print(f"RMSE:   ₸{rmse_tenge:,.0f}")
print(f"MAPE:   {mape_actual:.2%}")
print(f"\nСредняя цена: ₸{y_val_actual.mean():,.0f}")
print(f"MAE % от средней: {mae_tenge / y_val_actual.mean() * 100:.2f}%")

=== ENSEMBLE в Тенге ===
MAE:    ₸2,004,539
RMSE:   ₸5,562,920
MAPE:   16.75%

Средняя цена: ₸13,047,876
MAE % от средней: 15.36%


## 9. Feature Importance (CatBoost)

In [12]:
importance = pd.DataFrame({
    "feature": features,
    "importance": cat_model.get_feature_importance()
}).sort_values("importance", ascending=False)

print("\n=== Топ 15 важных фич ===")
print(importance.head(15).to_string(index=False))


=== Топ 15 важных фич ===
                 feature  importance
                 car_age   12.471825
    brand_target_encoded   10.848971
                  age_sq    8.700489
                    year    8.493514
        engine_volume_sq    8.399629
         engine_volume_l    7.678489
    model_target_encoded    6.186339
                brand_le    4.929879
                model_le    3.579902
              mileage_km    3.182520
body_type_target_encoded    3.036887
        mileage_per_year    2.845881
       generation_number    2.802494
           drive_type_le    2.788054
      generation_code_le    2.737172


## 10. Prediction Function

In [23]:
def predict_car_price(car_data, weights_ensemble=weights):
    """
    Предсказываем цену машины с доверительным интервалом.
    
    car_data: dict с параметрами машины
    """
    car_df = pd.DataFrame([car_data])
    
    # Обработка данных
    X_car, _, _, _, _ = process_data_v2(
        car_df, 
        is_train=False,
        target_encoders=target_enc,
        label_encoders=label_enc
    )
    
    X_car = X_car.reindex(columns=features, fill_value=-1)
    X_car = X_car.fillna(X_car.mean(numeric_only=True).to_dict())
    X_car = X_car.fillna(-1)
    
    # Предсказания
    cat_pred_log = cat_model.predict(X_car)[0]
    xgb_pred_log = xgb_model.predict(X_car)[0]
    lgb_pred_log = lgb_model.predict(X_car)[0]
    
    # Ансамбль
    ensemble_pred_log = weights_ensemble[0] * cat_pred_log + \
                        weights_ensemble[1] * xgb_pred_log + \
                        weights_ensemble[2] * lgb_pred_log
    
    # Обратное преобразование
    pred_price = np.expm1(ensemble_pred_log)
    
    # Доверительный интервал (более узкий благодаря ансамблю)
    # Используем остатки из валидации для оценки ошибки
    val_residuals = np.abs(y_val_actual - ensemble_pred_actual)
    std_error = np.std(val_residuals)
    
    # Интервалы: mean ± 1.96*std (95% confidence)
    low = pred_price * 0.85
    high = pred_price * 1.15
        
    return pred_price, low, high, {
        'cat': np.expm1(cat_pred_log),
        'xgb': np.expm1(xgb_pred_log),
        'lgb': np.expm1(lgb_pred_log),
    }


# Тестовый пример
test_car = {
    "brand": "Toyota",
    "model": "Camry",
    "year": 2021,
    "city": "Алматы",
    "mileage_km": 107800,
    "body_type": "Седан",
    "engine_volume_l": 2.0,
    "fuel_type": "бензин",
    "transmission": "Автомат",
    "drive_type": "Передний привод",
    "steering_wheel": "Слева",
    "color": "белый",
    "generation": "XV70",
    "generation_code": "XV70",
}

price, low, high, individual_preds = predict_car_price(test_car)

print(f"\n=== ТЕСТОВЫЙ ПРИМЕР ===")
print(f"Машина: {test_car['brand']} {test_car['model']} {test_car['year']}")
print(f"Параметры: {test_car['engine_volume_l']}L, {test_car['mileage_km']:,} км")
print(f"\n╔══════════════════════════════════╗")
print(f"║ Предсказанная цена: ₸{price:>15,.0f} ║")
print(f"║ Диапазон: ₸{low:>13,.0f} - {high:>13,.0f} ║")
print(f"╚══════════════════════════════════╝")
print(f"\nПредсказания отдельных моделей:")
print(f"  CatBoost: ₸{individual_preds['cat']:>15,.0f}")
print(f"  XGBoost:  ₸{individual_preds['xgb']:>15,.0f}")
print(f"  LightGBM: ₸{individual_preds['lgb']:>15,.0f}")


=== ТЕСТОВЫЙ ПРИМЕР ===
Машина: Toyota Camry 2021
Параметры: 2.0L, 107,800 км

╔══════════════════════════════════╗
║ Предсказанная цена: ₸     13,355,079 ║
║ Диапазон: ₸   11,351,818 -    15,358,341 ║
╚══════════════════════════════════╝

Предсказания отдельных моделей:
  CatBoost: ₸     12,183,753
  XGBoost:  ₸     13,769,719
  LightGBM: ₸     14,204,358


## 11. Model Comparison

In [14]:
comparison = pd.DataFrame({
    'Model': ['CatBoost', 'XGBoost', 'LightGBM', 'Ensemble'],
    'MAE (log)': [cat_mae, xgb_mae, lgb_mae, ensemble_mae],
    'RMSE (log)': [cat_rmse, xgb_rmse, lgb_rmse, ensemble_rmse],
    'R2 Score': [cat_r2, xgb_r2, lgb_r2, ensemble_r2],
    'MAPE (%)': [
        mean_absolute_percentage_error(y_val, cat_pred) * 100,
        mean_absolute_percentage_error(y_val, xgb_pred) * 100,
        mean_absolute_percentage_error(y_val, lgb_pred) * 100,
        ensemble_mape * 100
    ],
    'MAE (₸)': [
        mean_absolute_error(y_val_actual, np.expm1(cat_pred)),
        mean_absolute_error(y_val_actual, np.expm1(xgb_pred)),
        mean_absolute_error(y_val_actual, np.expm1(lgb_pred)),
        mae_tenge
    ]
})

print("\n=== СРАВНЕНИЕ МОДЕЛЕЙ ===")
print(comparison.round(4).to_string(index=False))


=== СРАВНЕНИЕ МОДЕЛЕЙ ===
   Model  MAE (log)  RMSE (log)  R2 Score  MAPE (%)      MAE (₸)
CatBoost     0.1583      0.2386    0.9373    1.0140 1938938.3562
 XGBoost     0.1773      0.2599    0.9256    1.1301 2392147.7929
LightGBM     0.1596      0.2401    0.9365    1.0240 1929624.1119
Ensemble     0.1590      0.2389    0.9371    1.0176 2004539.1172


,year,mileage_km,engine_volume_l,car_age,mileage_per_year,generation_number,engine_volume_sq,age_times_volume,age_sq,mileage_missing,...,body_type_le,fuel_type_le,transmission_le,drive_type_le,steering_wheel_le,color_le,generation_code_le,brand_target_encoded,model_target_encoded,body_type_target_encoded
2290,2005,168000.0,3.0,21,7636.363636,1,9.00,63.0,441,0,...,6,4,0,2,2,1,26,16.081737,15.916051,15.927665
3598,2020,80000.0,2.0,6,11428.571429,3,4.00,12.0,36,0,...,3,4,0,3,1,15,49,16.023855,16.217929,16.540715
439,2018,159000.0,2.8,8,17666.666667,0,7.84,22.4,64,0,...,2,0,0,3,1,3,31,16.068624,16.545276,16.678188
3511,2023,58000.0,2.0,3,14500.000000,5,4.00,6.0,9,0,...,3,4,0,3,1,1,83,16.022725,16.039718,16.540715
3683,2021,48000.0,5.7,5,8000.000000,2,32.49,28.5,25,0,...,7,4,0,3,1,1,82,16.079262,16.145062,16.457642
